Install Dependencies

In [1]:
# Installing all required packages
!pip install -U langchain-google-genai
!pip install langchain langchain-tavily
!pip install langchain-mcp-adapters
!pip install -qU langchain-community pypdf
!pip install -qU langchain-text-splitter
!pip install -qU langchain-huggingface sentence_transformers
!pip install -qU langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: Could not find a version that satisfies the requirement langchain-text-splitter (from versions: none)
ERROR: No matching distribution found for langchain-text-splitter
  

Imports and Environment Setup

In [2]:
import requests
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent

/tmp/ipykernel_1363/2362854158.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# Retrieve API keys from Colab userdata
google_api_key = userdata.get('gemini_api_key')
tavily_api_key = userdata.get('tavily_apikey')
rapid_api_key = userdata.get('rapid_apikey')

# Initialize shared Gemini model
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

# Initialize shared text splitter and embedding model
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dual RAG Pipeline (Processing Research Paper & Resume)

In [4]:
# --- 1. PROCESS THE RESEARCH PAPER ---
research_path = "/content/attention_is_all_you_need.pdf"  # Can be changed to any research PDF
research_loader = PyPDFLoader(research_path)
research_docs = research_loader.load()
research_splits = text_splitter.split_documents(research_docs)

research_vector_store = Chroma(
    collection_name="research_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_research_db",
)
research_vector_store.add_documents(documents=research_splits)


# --- 2. PROCESS THE RESUME ---
resume_path = "/content/RESUME.pdf"  # Upload your resume here
resume_loader = PyPDFLoader(resume_path)
resume_docs = resume_loader.load()
resume_splits = text_splitter.split_documents(resume_docs)

resume_vector_store = Chroma(
    collection_name="resume_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_resume_db",
)
resume_vector_store.add_documents(documents=resume_splits)

print(f"Successfully indexed {len(research_splits)} research chunks and {len(resume_splits)} resume chunks.")

Successfully indexed 52 research chunks and 4 resume chunks.


Defining Custom Tools (With Isolated Vector Queries)

In [5]:
@tool
def retrieve_from_research_pdf(query: str, k: int = 2) -> tuple:
    """Retrieve precise academic or technical information from the loaded research paper PDF."""
    retrieved_docs = research_vector_store.similarity_search(query, k=k)
    docs_content = ""
    for doc in retrieved_docs:
        docs_content += f"Source: {doc.metadata}\nContent: {doc.page_content}\n\n"
    return docs_content, retrieved_docs

@tool
def retrieve_from_resume(query: str, k: int = 2) -> tuple:
    """Retrieve information regarding the user's background, skills, experience, or projects from their resume PDF."""
    retrieved_docs = resume_vector_store.similarity_search(query, k=k)
    docs_content = ""
    for doc in retrieved_docs:
        docs_content += f"Source: {doc.metadata}\nContent: {doc.page_content}\n\n"
    return docs_content, retrieved_docs

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for live job listings requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\n[Tool Call] Searching jobs for: {skill} in {location}")
    url = "https://jsearch.p.rapidapi.com/search"
    headers = {"x-rapidapi-key": rapid_api_key, "x-rapidapi-host": "jsearch.p.rapidapi.com"}
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    jobs = data.get("data", [])

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

Dynamic Multi-Purpose System Prompt

In [6]:
system_prompt = """You are a highly adaptable, multi-purpose assistant capable of executing deep technical research, acting as a career advisor, and matching user resumes to live job openings.

You have access to these specific tools:
1. mcp_tavily (search tool): Use this to search the open web for broad market data, salary trends, updates, or events outside the files.
2. search_jobs: Use this to fetch active job postings based on skills and locations.
3. retrieve_from_research_pdf: Use this to extract technical or scientific data from the uploaded research document.
4. retrieve_from_resume: Use this to understand the user's core competencies, skills, experience, and professional goals from their resume.

Routing Strategy & Rules:
- When asked technical questions about core academic concepts -> Use retrieve_from_research_pdf.
- When asked about what jobs fit the user, or what skills are listed in their profile -> Use retrieve_from_resume first to extract user context, then use search_jobs or mcp_tavily to find corresponding market listings.
- If a hybrid request is made (e.g., 'Look at my resume, find out if I have the skills mentioned in the research paper, and look for jobs for me') -> Coordinate your tools sequentially. First check the paper, then check the resume, then find jobs.

Formatting Guidelines:
- Present job results cleanly and directly. Do not use markdown syntax specifically for job listing text blocks.
- [Unverified] For career behavior or job placement claims, note that this is expected behavior, not guaranteed.
- If information is not present across tools, explicitly state what is missing. Do not hallucinate data.
"""

MCP Setup and Agent Core Configuration

In [7]:
# Configure the MCP Tavily Client
client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_api_key}",
        }
    }
)

async def run_unified_agent(user_query: str):
    mcp_tools = await client.get_tools()

    # Combine all specific tools into one environment
    all_tools = mcp_tools + [search_jobs, retrieve_from_research_pdf, retrieve_from_resume]

    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=system_prompt,
        debug=True
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_query}]
    })

    print("\n--- AGENT RESPONSE ---")
    print(response["messages"][-1].content[0]["text"])

Agent Execution

In [9]:
# Asking a combined multi-part query
query = """
1. What are the key findings or architectures explained in the research paper?
2. Based on my resume, do I have the prerequisite skills to work on that architecture?
3. Find 2 live job openings in Bangalore that match the skills found on my resume.
"""

await run_unified_agent(query)

[values] {'messages': [HumanMessage(content='\n1. What are the key findings or architectures explained in the research paper?\n2. Based on my resume, do I have the prerequisite skills to work on that architecture?\n3. Find 2 live job openings in Bangalore that match the skills found on my resume.\n', additional_kwargs={}, response_metadata={}, id='7fd427da-47fd-492d-9205-5bba128ed35c')]}


[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_research_pdf', 'arguments': '{"query": "key findings or architectures explained in the research paper"}'}, '__gemini_function_call_thought_signatures__': {'2520fa53-325a-4038-b1e9-3ac0cefeaf4e': 'CqMHARFNMg/NI7MYpTT+P30bPE3LDWsBQgjdSOBOSHMszkTiPpSYVWq22NLGx1fn+3hLNWGrdHxbL10r1nQvocw9oLD1gXxTgppS8+rnoQYmK5xaAxAeQ+VPzI24YvYDFGbicT/yyP5kqts5DYq52/bgvKETTUklDL0OOx8tNSpQYkNVrN1VrRctIoIurRU9ssPOCreuwtf3u+e5oOWJhI7baFGh4Tm13rNsvrXcgKfD76DzQ8guoE9B1/Nvehhj9p2uwLUh8HJCXNOFkKuo6x9fid3XP9yfO+eA42iOtWlU4UzCe7WATBfN8nUhZVS1rkhdAvc2H1f8ItED2jSvarZQ4f3+hnrKHwwD0VOInbXgtmq31CttSFvXcjoXW3rL0tq1v0bRx16UrFxgAEyyJXitBrm4RaAefjPzJYO1XT/zu+N2qn8iIrHJkljCCZRUL9+cGYHGPYlmbWzVPgnhr6KXMJfip18JdCs8nrZ1oyQ2GRVT40dxYmEnU2TpDGPUyKFTwp6Nx8QAWzsEDU6PGdtEgzH3r2hqIngKeqxCCEtDgdkD/vupq/wD40ADDi+E9noFxOlvEr6A73Xt5XzZNuhx9UWR5TJXKfx+rkXXnjUc9BqQtMhDJI1o+1CZBZE/7b9GI0Og7/6OpYBqsiUEnvZrUp1idoc0mlCUU03t1bB6WAxO

[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_resume', 'arguments': '{"query": "skills related to machine learning, deep learning, natural language processing, neural networks, or Transformer architecture"}'}, '__gemini_function_call_thought_signatures__': {'6b5197b0-0bdb-47f2-ae99-5315897646d2': 'CsEIARFNMg9amSR9y+D4YoEeMvz+k/l0UQ9erO+RbTFAOPSj7g+z4x7qKhD70Yj8rd5/LPTNTHMQvRDz2e9oiekt1JLkmE1qUsW0p3iXOM8H96kMwheD0ntf17/fMSpbttwQP11ECkMMNw8WGxIMOPxhYY3YReWphzH+5Nq5bdq4xRZsalRF/8eAY9xhwKg0JqXCOdAmVAGdJCyy9rma3OwGUmpZysnXU8Tz3UFy8JlAOXPOlLZoBbY8MgJryelUt6SV7ueJqL0q1IdubJU1148dsSJa173xQxLCwV+i/7SvS1CcsG3aTKjmnsLMC3WG8Y6H1feQp7vslmVR7h/zMgotV93++YGfSDhqe+xyjuVQpCRFKKpgV+mezB8t2gpp4Tj8cTW+g6/Tz7HJ04BPeUlzznbIl4M3Wj44VfmUqOSjJFQ++tKxPt0FJyK9hZdZOwA0aA4/iA4v5OV2KG79RuwELZ4fqIv+ap7YDnrD+5ZYV94xA/Dzcb3auIaZv1SFl+kV6JBERSGYX14KKdvhBcLPYOSHlqqwjIbPG+a1h+heixDHpg8JBrQaGOJOou7tD5ivovIJ17VxENFI5ZC/6GUnejpTksr0SCy4suNekRsEKBaQSCZNi5q

[updates] {'tools': {'messages': [ToolMessage(content=[], name='search_jobs', id='c53054ea-f1ee-4995-9faf-24fca4f38963', tool_call_id='76ab1997-c07e-4dd5-be6c-f76fff7bfd7b')]}}
[values] {'messages': [HumanMessage(content='\n1. What are the key findings or architectures explained in the research paper?\n2. Based on my resume, do I have the prerequisite skills to work on that architecture?\n3. Find 2 live job openings in Bangalore that match the skills found on my resume.\n', additional_kwargs={}, response_metadata={}, id='7fd427da-47fd-492d-9205-5bba128ed35c'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_from_research_pdf', 'arguments': '{"query": "key findings or architectures explained in the research paper"}'}, '__gemini_function_call_thought_signatures__': {'2520fa53-325a-4038-b1e9-3ac0cefeaf4e': 'CqMHARFNMg/NI7MYpTT+P30bPE3LDWsBQgjdSOBOSHMszkTiPpSYVWq22NLGx1fn+3hLNWGrdHxbL10r1nQvocw9oLD1gXxTgppS8+rnoQYmK5xaAxAeQ+VPzI24YvYDFGbicT/yyP5kqts5DYq52/bgvKE

[updates] {'tools': {'messages': [ToolMessage(content='[{"title": "AI Engineering Internship", "company": "epiFi technologies pvt. ltd.", "location": "Bengaluru", "apply_link": "https://unstop.com/internships/ai-engineering-internship-epifi-technologies-pvt-ltd-1710737"}, {"title": "Principal Product AI Data Engineer", "company": "Clarivate", "location": "Bengaluru", "apply_link": "https://careers.clarivate.com/job/JREQ135703/Principal-Product-AI-Data-Engineer"}, {"title": "Data Scientist- GEN AI", "company": "Intellikart Ventures LLP", "location": "Bengaluru", "apply_link": "https://cutshort.io/job/Data-Scientist-GEN-AI-Bengaluru-Bangalore-Intellikart-Ventures-LLP-LDo13QkG"}, {"title": "AI\u200b/ML", "company": "Infosys", "location": "Bengaluru", "apply_link": "https://www.learn4good.com/jobs/bangalore/india/info_technology/5240266543/e/"}, {"title": "Prompt Engineering Intern", "company": "foundit", "location": "Bengaluru", "apply_link": "https://www.foundit.in/zuno/app/job/prompt-en